# HerBERT-large — domknięcie siatki poolingu: wariant TW+rzadkie(CE)

Dolicza wariant TW+rzadkie(CE) — odjęcie GoEmotions-PL rozstrzyga, czy szkodzi sam dobór wierszy pod kątem etykiety, czy dołączony z nimi korpus tłumaczony maszynowo.

**Wymaga:** GPU T4×2, Internet ON, dataset `pl-emotion-processed`.

In [ ]:
# transformers <5.2 — w 5.2 usunięto warmup_ratio z TrainingArguments.
# herbert_large_epochs padł na tym 10.08.2026). Górne ograniczenie utrzymuje
# recepturę identyczną z wcześniejszymi runami tej kampanii.
!pip install -q -U "transformers>=4.44,<5.2" "datasets>=2.20" accelerate 2>/dev/null
import torch, transformers
print(transformers.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
import os, gc, glob, shutil, time, warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.special import expit
from sklearn.metrics import (f1_score, hamming_loss, jaccard_score, accuracy_score,
                             precision_score, recall_score)
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
warnings.filterwarnings("ignore")
RANDOM_STATE=42; torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
LABELS9=EMOTIONS+["sarkazm"]
RARE=["strach","zaufanie","smutek"]
OUT="/kaggle/working"
MODEL_NAME="allegro/herbert-large-cased"; MAX_LEN, EPOCHS, BATCH, LR = 128, 3, 16, 2e-5
RESULT_CSV=f"{OUT}/result.csv"

In [ ]:
def find_csv(n):
    h=glob.glob(f"/kaggle/input/**/{n}",recursive=True)
    if not h: raise FileNotFoundError(f"{n} — dołącz dataset pl-emotion-processed")
    return h[0]

def load(prefix, text_col):
    out={}
    for sp in ("train","val","test"):
        d=pd.read_csv(find_csv(f"{prefix}_{sp}.csv")).reset_index(drop=True)
        d["__text"]=d[text_col].fillna("")
        out[sp]=d
    return out

TW=load("twitteremo","tekst")
CE=load("clarin_emo","tekst")
GO=load("go_emotions","text_pl")
# pliki po korekcie zawierają WYŁĄCZNIE kolumnę text_lt — etykiety biorą się
# z oryginalnych splitów (ten sam porządek wierszy), jak w eksperymencie 14
LT={}
for _sp in ("train","val","test"):
    _base=TW[_sp].copy()
    _base["__text"]=pd.read_csv(find_csv(f"lt_twitteremo_{_sp}.csv"))["text_lt"].fillna("").values
    LT[_sp]=_base
y_val,y_test=TW["val"][EMOTIONS].values,TW["test"][EMOTIONS].values
print("TW",len(TW["train"]),"| CE",len(CE["train"]),"| GO",len(GO["train"]),"| LT",len(LT["train"]))

In [ ]:
def evaluate(yt,yp):
    return {"f1_macro":f1_score(yt,yp,average="macro",zero_division=0),"f1_micro":f1_score(yt,yp,average="micro",zero_division=0),
            "f1_weighted":f1_score(yt,yp,average="weighted",zero_division=0),"precision_macro":precision_score(yt,yp,average="macro",zero_division=0),
            "recall_macro":recall_score(yt,yp,average="macro",zero_division=0),"hamming_loss":hamming_loss(yt,yp),
            "jaccard_macro":jaccard_score(yt,yp,average="macro",zero_division=0),"subset_accuracy":accuracy_score(yt,yp)}

def find_optimal_thresholds(yt,yp):
    thr=np.full(yt.shape[1],0.5)
    for i in range(yt.shape[1]):
        bf,bt=0.0,0.5
        for t in np.arange(0.05,0.95,0.01):
            f=f1_score(yt[:,i],(yp[:,i]>=t).astype(int),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[i]=bt
    return thr

def f1_macro_ci(yt,yp,n_boot=1000,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed); n=len(yt); base=f1_score(yt,yp,average="macro",zero_division=0)
    b=[f1_score(yt[i],yp[i],average="macro",zero_division=0) for i in (rng.integers(0,n,n) for _ in range(n_boot))]
    lo,hi=np.percentile(b,[2.5,97.5]); return base,lo,hi

In [ ]:
tok=AutoTokenizer.from_pretrained(MODEL_NAME)

class WeightedTrainer(Trainer):
    def __init__(self,*a,pos_weight=None,**k): super().__init__(*a,**k); self.pw=pos_weight
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        lab=inputs.pop("labels"); out=model(**inputs)
        loss=F.binary_cross_entropy_with_logits(out.logits.float(),lab.float(),pos_weight=self.pw.to(out.logits.device))
        return (loss,out) if return_outputs else loss

def to_ds(texts, labels_arr):
    d=Dataset.from_dict({"text":list(texts),"labels":labels_arr.astype("float32").tolist()})
    return d.map(lambda b: tok(b["text"],truncation=True,max_length=MAX_LEN),batched=True,remove_columns=["text"])

def run(name, train_texts, train_y, val_texts, val_y, test_texts, test_y, labels):
    """Fine-tune HerBERT-large on the given train set; thresholds on val, metrics on test."""
    done=set()
    if os.path.exists(RESULT_CSV): done=set(pd.read_csv(RESULT_CSV)["warunek"])
    if name in done:
        print(f"== {name}: już policzony, pomijam"); return
    t0=time.time(); print(f"\n=== {name} (n_train={len(train_texts)}, etykiet={len(labels)}) ===",flush=True)

    pos=train_y.sum(0); neg=len(train_y)-pos
    pw=torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,10.0),dtype=torch.float32)

    ds_tr,ds_va,ds_te=to_ds(train_texts,train_y),to_ds(val_texts,val_y),to_ds(test_texts,test_y)
    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,num_labels=len(labels),problem_type="multi_label_classification")
    args=TrainingArguments(output_dir=f"{OUT}/ckpt_{name}",eval_strategy="epoch",save_strategy="epoch",save_total_limit=1,
        load_best_model_at_end=True,metric_for_best_model="f1_macro",greater_is_better=True,per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=32,gradient_accumulation_steps=2,gradient_checkpointing=True,num_train_epochs=EPOCHS,
        learning_rate=LR,warmup_ratio=0.1,weight_decay=0.01,fp16=True,logging_steps=200,report_to="none",seed=RANDOM_STATE)
    cm=lambda p:{"f1_macro":f1_score(p.label_ids.astype(int),(expit(p.predictions)>=0.5).astype(int),average="macro",zero_division=0)}
    trainer=WeightedTrainer(model=model,args=args,train_dataset=ds_tr,eval_dataset=ds_va,
        data_collator=DataCollatorWithPadding(tok),compute_metrics=cm,pos_weight=pw,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()

    p_val=expit(trainer.predict(ds_va).predictions); p_test=expit(trainer.predict(ds_te).predictions)
    thr=find_optimal_thresholds(val_y,p_val); pred=(p_test>=thr).astype(int)

    # metryki liczone ZAWSZE na 8 emocjach, żeby wariant 9-etykietowy był porównywalny
    m=evaluate(test_y[:,:8],pred[:,:8]); base,lo,hi=f1_macro_ci(test_y[:,:8],pred[:,:8])
    m.update({"warunek":name,"n_train":len(train_texts),"ci_low":round(lo,3),"ci_high":round(hi,3)})
    for c in RARE:
        i=EMOTIONS.index(c); m[f"recall_{c}"]=recall_score(test_y[:,i],pred[:,i],zero_division=0)
    for i,c in enumerate(EMOTIONS):
        m[f"f1_{c}"]=f1_score(test_y[:,i],pred[:,i],zero_division=0)
    m["f1_rare_avg"]=float(np.mean([m[f"f1_{c}"] for c in RARE]))
    m["f1_frequent_avg"]=float(np.mean([m[f"f1_{c}"] for c in EMOTIONS if c not in RARE]))
    if len(labels)==9:
        m["f1_sarkazm"]=f1_score(test_y[:,8],pred[:,8],zero_division=0)
        m["precision_sarkazm"]=precision_score(test_y[:,8],pred[:,8],zero_division=0)
        m["recall_sarkazm"]=recall_score(test_y[:,8],pred[:,8],zero_division=0)

    # wariant 9-etykietowy dokłada kolumny, więc dopisywanie bez wyrównania psuje CSV
    row=pd.DataFrame([m])
    if os.path.exists(RESULT_CSV):
        row=pd.concat([pd.read_csv(RESULT_CSV),row],ignore_index=True)
    row.to_csv(RESULT_CSV,index=False)
    np.save(f"{OUT}/proba_test_{name}.npy",p_test)
    np.save(f"{OUT}/proba_val_{name}.npy",p_val); np.save(f"{OUT}/thr_{name}.npy",thr)
    print(f"   F1-Macro(8 emocji)={m['f1_macro']:.3f} [{lo:.3f},{hi:.3f}]  ({time.time()-t0:.0f}s)",flush=True)

    # punkt kontrolny (4 GB!) jest już niepotrzebny — do dalszych analiz służą
    # zapisane macierze prawdopodobieństw. Bez tego 6 warunków przekracza
    # limit 20 GB katalogu /kaggle/working i kernel pada w połowie.
    shutil.rmtree(args.output_dir, ignore_errors=True)
    del trainer, model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- POOLING (eksperyment 21 na large): czy pojemność modelu odwraca wniosek z wersji base? ---
def pool(*dfs):
    return pd.concat([d[["__text"]+EMOTIONS] for d in dfs],ignore_index=True)

rare_ce=CE["train"][CE["train"][RARE].sum(1)>0]
print(f"wierszy CE z klasą rzadką: {len(rare_ce)}")
for nm,df in [("pool_TW+rare(CE)", pool(TW["train"],rare_ce))]:
    run(nm, df["__text"].tolist(), df[EMOTIONS].values,
        TW["val"]["__text"].tolist(), y_val, TW["test"]["__text"].tolist(), y_test, EMOTIONS)

In [ ]:
res=pd.read_csv(RESULT_CSV)
print("Punkt odniesienia: HerBERT-large-full na czystym TW = 0,590\n")
cols=[c for c in ["warunek","n_train","f1_macro","ci_low","ci_high","f1_micro","f1_sarkazm"] if c in res.columns]
display(res[cols].round(3))